In [ ]:

"""
Script: two-step clustering + local classifiers (COSINE via L2-normalized Euclidean)
- K-sweep + FCM (m=FUZZINESS_M), centers L2-normalized (để xấp xỉ cosine)
- Per-cluster local models: chỉ train khi đủ dữ liệu & đủ lớp (khoa học)
- Fallback KHÔNG còn "đa số cụm": dùng Mixture-of-Experts (MoE)
    + Expert GLOBAL: model train toàn bộ train set -> P_global(y|x)
    + Expert LOCAL_i: model chỉ của cụm i (nếu đủ dữ liệu) -> P_i(y|x_sub)
    + Gating: membership mềm của FCM (có mũ LAMBDA_GATE) + trọng số nền global
    + Hiệu chỉnh PRIOR theo cụm (label shift) với shrinkage về prior toàn cục
- Không lưu model đã fit xuống đĩa (chỉ lưu metadata/configs/centers/label_maps)
- Xuất đầy đủ báo cáo, biểu đồ như bản gốc

"""

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['PYTHONHASHSEED'] = '42'

import random
random.seed(42)

import numpy as np
np.random.seed(42)

import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize, normalize
import sklearn.metrics as metrics
from scipy.spatial.distance import cdist
import skfuzzy as fuzz
import joblib
import warnings
import itertools

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# =========================================================
#               MoE / FALLBACK HYPER-PARAMETERS
# =========================================================
# TINH CHỈNH GỢI Ý:
# - Nếu local model quá "ồn", tăng MIN_* để yêu cầu dữ liệu cụm nhiều hơn.
# - Nếu muốn gating tập trung vào cụm gần nhất hơn, tăng LAMBDA_GATE (2.0 -> 3.0).
# - Nếu cụm nhỏ làm lệch prior quá mạnh, tăng PRIOR_SMOOTH_TAU để kéo về prior global.
# - Nếu chi phí dự báo lớn, tăng MEMBERSHIP_THRESH (vd 0.02~0.05) để bỏ qua cụm weight quá nhỏ.
MIN_TOTAL_SAMPLES      = 30     # tối thiểu tổng mẫu trong cụm để xem xét train local
MIN_SAMPLES_PER_CLASS  = 5      # tối thiểu mỗi lớp (và cần >=2 lớp) trong cụm để train local
FUZZINESS_M            = 2      # m của FCM (thường dùng 2)
LAMBDA_GATE            = 2.0    # mũ làm "sắc" membership khi làm weight: w_i = u_i ** LAMBDA_GATE
GLOBAL_BASE_WEIGHT     = 0.5    # trọng số nền luôn dành cho expert global (>=0). 0.5 là cân bằng.
PRIOR_SMOOTH_TAU       = 10.0   # độ trơn/shrinkage ưu tiên kéo prior cụm về prior toàn cục
MEMBERSHIP_THRESH      = 1e-3   # ngưỡng nhỏ để bỏ qua cụm gần như không liên quan
EPS_SAFE               = 1e-12  # tránh chia cho 0/ log(0)

# ---------------------------
# Utility & plotting helpers
# ---------------------------
def ensure_dir(path):
    """Ensure directory exists for a file path (create parent if needed)."""
    d = os.path.dirname(path)
    if d and not os.path.exists(d):
        os.makedirs(d, exist_ok=True)

def plot_bar_per_cluster(df_cluster_both, current_result_dir, model_name, n_clusters):
    """Bar charts: per-cluster scores and counts."""
    out_prefix = os.path.join(current_result_dir, 'plots')
    os.makedirs(out_prefix, exist_ok=True)

    dfcb = df_cluster_both.sort_values('cluster').reset_index(drop=True)
    clusters = dfcb['cluster'].astype(int).tolist()
    accs = dfcb.get('accuracy', pd.Series([np.nan]*len(dfcb))).fillna(0).tolist()
    f1s = dfcb.get('f1_macro', pd.Series([np.nan]*len(dfcb))).fillna(0).tolist()
    pre = dfcb.get('precision_macro', pd.Series([np.nan]*len(dfcb))).fillna(0).tolist()
    rec = dfcb.get('recall_macro', pd.Series([np.nan]*len(dfcb))).fillna(0).tolist()
    n_train = dfcb.get('n_train_samples', pd.Series([0]*len(dfcb))).fillna(0).astype(int).tolist()
    n_test = dfcb.get('n_test_samples', pd.Series([0]*len(dfcb))).fillna(0).astype(int).tolist()

    x = np.arange(len(clusters))
    width = 0.2

    plt.figure(figsize=(max(8, len(clusters)*0.6), 5))
    plt.bar(x - 1.5*width, accs, width)
    plt.bar(x - 0.5*width, f1s, width)
    plt.bar(x + 0.5*width, pre, width)
    plt.bar(x + 1.5*width, rec, width)
    plt.xticks(x, clusters)
    plt.xlabel('Cluster id')
    plt.ylabel('Score')
    plt.title(f'Per-cluster scores — {model_name} — K={n_clusters}')
    plt.legend(['Accuracy','F1-macro','Precision-macro','Recall-macro'])
    plt.tight_layout()
    ppath = os.path.join(out_prefix, f'per_cluster_scores_K{n_clusters}_{model_name}.png')
    plt.savefig(ppath, dpi=200)
    plt.close()

    plt.figure(figsize=(max(8, len(clusters)*0.6), 4))
    plt.bar(x - 0.2, n_train, width=0.4)
    plt.bar(x + 0.2, n_test, width=0.4)
    plt.xticks(x, clusters)
    plt.xlabel('Cluster id')
    plt.ylabel('Number of samples')
    plt.title(f'Per-cluster sample counts — {model_name} — K={n_clusters}')
    plt.legend(['n_train','n_test'])
    plt.tight_layout()
    ppath2 = os.path.join(out_prefix, f'per_cluster_counts_K{n_clusters}_{model_name}.png')
    plt.savefig(ppath2, dpi=200)
    plt.close()

def plot_confusion_matrix(cm, target_names, current_result_dir, model_name, n_clusters, normalize=False):
    """Plot and save confusion matrix (normalized or raw)."""
    out_prefix = os.path.join(current_result_dir, 'plots')
    os.makedirs(out_prefix, exist_ok=True)
    if normalize:
        with np.errstate(all='ignore'):
            cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
            cm_plot = np.nan_to_num(cm_norm)
    else:
        cm_plot = cm

    plt.figure(figsize=(8,6))
    plt.imshow(cm_plot, interpolation='nearest', aspect='auto')
    plt.title(f'Confusion matrix — {model_name} — K={n_clusters} {"(norm)" if normalize else ""}')
    plt.colorbar()
    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=90)
    plt.yticks(tick_marks, target_names)
    fmt = '.2f' if normalize else 'd'
    for i, j in itertools.product(range(cm_plot.shape[0]), range(cm_plot.shape[1])):
        val = cm_plot[i, j]
        if normalize:
            plt.text(j, i, format(val, fmt), ha="center", va="center", fontsize=8)
        else:
            plt.text(j, i, int(val), ha="center", va="center", fontsize=8)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    p = os.path.join(out_prefix, f'confusion_matrix_K{n_clusters}_{model_name}{"_norm" if normalize else ""}.png')
    plt.savefig(p, dpi=200)
    plt.close()

def plot_over_k(df_report, RESULT_DIR):
    """Plot trends vs K: PC/CE/XB, Accuracy per model, F1_macro per model, time metrics."""
    outdir = os.path.join(RESULT_DIR, 'summary_plots')
    os.makedirs(outdir, exist_ok=True)
    df = df_report.sort_values('Số cụm')

    plt.figure()
    if 'PC' in df.columns: plt.plot(df['Số cụm'], df['PC'], marker='o', label='PC')
    if 'CE' in df.columns: plt.plot(df['Số cụm'], df['CE'], marker='o', label='CE')
    if 'XB' in df.columns: plt.plot(df['Số cụm'], df['XB'], marker='o', label='XB')
    plt.xlabel('K (số cụm)'); plt.ylabel('Score'); plt.title('Cluster quality metrics vs K')
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(outdir, 'cluster_quality_vs_K.png'), dpi=200); plt.close()

    acc_cols = [c for c in df.columns if c.startswith('Accuracy_')]
    if acc_cols:
        plt.figure()
        for c in acc_cols:
            plt.plot(df['Số cụm'], df[c], marker='o', label=c.replace('Accuracy_',''))
        plt.xlabel('K (số cụm)'); plt.ylabel('Accuracy'); plt.title('Accuracy (per model) vs K')
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(outdir, 'accuracy_per_model_vs_K.png'), dpi=200); plt.close()

    f1_macro_cols = [c for c in df.columns if c.startswith('F1_Macro_')]
    if f1_macro_cols:
        plt.figure()
        for c in f1_macro_cols:
            plt.plot(df['Số cụm'], df[c], marker='o', label=c.replace('F1_Macro_',''))
        plt.xlabel('K (số cụm)'); plt.ylabel('F1 (Macro)'); plt.title('F1-macro (per model) vs K')
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(outdir, 'f1_macro_per_model_vs_K.png'), dpi=200); plt.close()

    time_cols = [c for c in df.columns if 'Time_Total_Predict_Per_Sample_s_' in c or 'Time_Assign_One_Sample_s_' in c]
    if time_cols:
        plt.figure()
        for c in sorted(time_cols):
            plt.plot(df['Số cụm'], df[c], marker='o',
                     label=c.replace('Time_Total_Predict_Per_Sample_s_','Ttp:').replace('Time_Assign_One_Sample_s_','Tassign:'))
        plt.xlabel('K (số cụm)'); plt.ylabel('Time (s)'); plt.title('Prediction time metrics vs K')
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(outdir, 'time_metrics_vs_K.png'), dpi=200); plt.close()

# ---------------------------
# Metrics helpers
# ---------------------------
def calculate_partition_entropy(u):
    epsilon = 1e-9; u_safe = u + epsilon
    return -np.sum(u_safe * np.log2(u_safe)) / u.shape[1]

def calculate_xie_beni_normed(data_norm, u, centers_norm, m):
    d_sq = (cdist(data_norm, centers_norm) ** 2)
    numerator = np.sum((u ** m) * d_sq)
    dist_matrix = cdist(centers_norm, centers_norm) + np.inf * np.eye(len(centers_norm))
    min_dist_sq = np.min(dist_matrix) ** 2
    if min_dist_sq < 1e-12:
        return np.inf
    denominator = data_norm.shape[0] * min_dist_sq
    return numerator / denominator if denominator != 0 else np.inf

# ---------------------------
# MoE helpers
# ---------------------------
def fuzzy_membership_from_distances(D, m=2, eps=1e-12):
    """
    Convert distance matrix D (n_samples x n_clusters) -> fuzzy memberships U.
    FCM formula: u_ik = 1 / sum_j (d_ik / d_jk)^(2/(m-1)).
    Xử lý d≈0: nếu có d=0, gán 1 cho cụm đó (nếu nhiều cụm đồng thời d≈0, chia đều).
    """
    n, k = D.shape
    U = np.zeros_like(D, dtype=float)
    D_safe = D + eps
    power = 2.0 / (m - 1.0)

    # Hàng có d=0
    zero_mask = (D <= eps)
    any_zero = zero_mask.any(axis=1)
    if np.any(any_zero):
        idx = np.where(any_zero)[0]
        for r in idx:
            cols = np.where(zero_mask[r])[0]
            U[r, cols] = 1.0 / len(cols)

    # Hàng bình thường
    idx_norm = np.where(~any_zero)[0]
    if idx_norm.size:
        Dn = D_safe[idx_norm]
        for i in range(k):
            ratio = (Dn[:, [i]] / Dn) ** power
            U[idx_norm, i] = 1.0 / np.sum(ratio, axis=1)

    # Chuẩn hóa
    row_sum = U.sum(axis=1, keepdims=True) + eps
    U = U / row_sum
    return U

def proba_local_to_global(proba_local, local_classes_in_global, n_classes):
    """
    Map ma trận proba của local model (trên subset lớp) về vector đủ n_classes theo chỉ số lớp global.
    local_classes_in_global: list các chỉ số lớp (global) theo thứ tự cột trong proba_local
    """
    out = np.zeros((proba_local.shape[0], n_classes), dtype=float)
    for j, g in enumerate(local_classes_in_global):
        out[:, g] = proba_local[:, j]
    return out

# ---------------------------
# Main experiment
# ---------------------------
RESULT_DIR = 'result_3ML_cosine_moe_fair_test_time'
os.makedirs(RESULT_DIR, exist_ok=True)

# Load data
DATA_CSV = '/home/nhannv02/Hello/twostepcluster/data/raw/data1.csv'
if not os.path.exists(DATA_CSV):
    raise FileNotFoundError(f"Data file not found: {DATA_CSV}")
df = pd.read_csv(DATA_CSV)
if 'label' not in df.columns:
    raise ValueError("CSV must contain a 'label' column.")
X = df.drop('label', axis=1).values
y = df['label'].values

le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_
n_classes = len(class_names)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=10, stratify=y_encoded
)

# Standard scaler (for classifiers)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(RESULT_DIR, 'data_scaler.joblib'))

# Classifiers to try (templates)
classifiers_to_test = {
    "RandomForest": RandomForestClassifier(n_estimators=15, random_state=10, n_jobs=-1),
    "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=10),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=15, random_state=10, n_jobs=-1)
}

values, counts = np.unique(y_train, return_counts=True)
most_frequent_class = int(values[np.argmax(counts)])  # (giữ lại để phòng hờ, thực tế MoE không dùng)

overall_results = []

# K sweep
cluster_range = range(2, 10)
eps = 1e-12  # small value for normalize stability

for n_clusters in cluster_range:
    print("\n" + "#"*60 + f"\n# START K={n_clusters} (FCM using COSINE via L2-normalize)\n" + "#"*60)

    # START TOTAL TRAINING TIME MEASUREMENT
    start_time_total_training = time.perf_counter()

    # Normalize train data per-sample (unit vectors)
    X_train_norm = normalize(X_train_scaled + eps, norm='l2', axis=1)
    X_train_T = X_train_norm.T  # cmeans expects shape (features, samples)

    # Run FCM
    start_time_clustering_train = time.perf_counter()
    try:
        cntr, u, _, _, _, _, fpc = fuzz.cluster.cmeans(
            X_train_T,
            n_clusters,
            FUZZINESS_M,     # dùng m đã cấu hình
            error=0.005,
            maxiter=1000,
            init=None,
            seed=42
        )
    except Exception as e:
        print("FCM error:", e)
        continue
    if np.isnan(cntr).any():
        print("FCM returned NaN centers; skip this K.")
        continue
    time_clustering_train = time.perf_counter() - start_time_clustering_train
    print(f"Clustering done in {time_clustering_train:.4f}s")

    # Ensure centers normalized
    cntr_norm = normalize(cntr + eps, norm='l2', axis=1)

    # Evaluate clustering
    pc_metric = fpc
    ce_metric = calculate_partition_entropy(u)
    xb_metric = calculate_xie_beni_normed(X_train_norm, u.T, cntr_norm, FUZZINESS_M)
    print(f"PC={pc_metric:.6f}, CE={ce_metric:.6f}, XB={xb_metric:.6f}")
    cluster_membership_train = np.argmax(u, axis=0)

    cluster_summary = {
        'Số cụm': n_clusters, 'PC': float(pc_metric), 'CE': float(ce_metric),
        'XB': float(xb_metric), 'Time_Clustering_Train_s': float(time_clustering_train),
        'Time_Total_Training_s': float(time_clustering_train)  # Initialize with clustering time, will be updated later
    }

    # Save centers for this K
    centers_dir = os.path.join(RESULT_DIR, f'{n_clusters}_clusters')
    os.makedirs(centers_dir, exist_ok=True)
    np.save(os.path.join(centers_dir, 'cluster_centers_norm.npy'), cntr_norm)

    # For each classifier type
    for model_name, model_template in classifiers_to_test.items():
        print("\n" + "="*40 + f"\nModel: {model_name}\n" + "="*40)
        current_result_dir = os.path.join(RESULT_DIR, f'{n_clusters}_clusters', model_name)
        os.makedirs(current_result_dir, exist_ok=True)
        np.save(os.path.join(current_result_dir, 'cluster_centers_norm.npy'), cntr_norm)

        # Prepare label maps and model config metadata
        label_maps = {}
        model_configs = {}
        cluster_training_rows = []
        model_template_params = model_template.get_params()
        joblib.dump(model_template_params, os.path.join(current_result_dir, 'model_template_params.joblib'))

        trained_cluster_flags = {}
        # -------------------------------
        # PHÂN TÍCH KHẢ NĂNG TRAIN LOCAL
        # (khoa học: đủ tổng & đủ mẫu mỗi lớp cho ≥2 lớp)
        # -------------------------------
        for i in range(n_clusters):
            cluster_indices = np.where(cluster_membership_train == i)[0]
            n_samples_in_cluster = len(cluster_indices)
            n_unique_labels = 0
            unique_labels = []
            trained_flag = False

            if n_samples_in_cluster >= MIN_TOTAL_SAMPLES:
                y_cluster_train_global = y_train[cluster_indices]
                counts_ci = np.bincount(y_cluster_train_global, minlength=n_classes)
                if (counts_ci >= MIN_SAMPLES_PER_CLASS).sum() >= 2:
                    unique_labels = np.where(counts_ci > 0)[0]
                    n_unique_labels = len(unique_labels)
                    local_label_map = {g: l for l, g in enumerate(unique_labels)}
                    label_maps[i] = local_label_map
                    model_configs[i] = model_template_params
                    trained_flag = True

            cluster_training_rows.append({
                'cluster': int(i),
                'n_train_samples': int(n_samples_in_cluster),
                'n_unique_train_labels': int(n_unique_labels),
                'trained_model_available': bool(trained_flag),
                'model_params_saved': bool(trained_flag)
            })
            trained_cluster_flags[i] = trained_flag

        # Save metadata
        if label_maps:
            joblib.dump(label_maps, os.path.join(current_result_dir, 'label_maps.joblib'))
        joblib.dump(model_configs, os.path.join(current_result_dir, 'model_configs_per_cluster.joblib'))
        df_cluster_train = pd.DataFrame(cluster_training_rows)
        df_cluster_train.to_csv(os.path.join(current_result_dir, 'cluster_train_summary.csv'),
                                index=False, float_format='%.6f')

        # ---------------------
        # PREDICTION (MoE + prior correction)
        # ---------------------
        print("Predicting on test set (Mixture-of-Experts + prior correction fallback)...")
        start_time_total_prediction = time.perf_counter()

        X_test_norm = normalize(X_test_scaled + eps, norm='l2', axis=1)

        # đo thời gian assign 1 mẫu (để báo cáo)
        start_time_assign_single_point = time.perf_counter()
        if X_test_norm.shape[0] > 0:
            _ = cdist(X_test_norm[[0]], cntr_norm)
            _ = np.argmin(_, axis=1)
        time_assign_single_point = time.perf_counter() - start_time_assign_single_point

        # khoảng cách & membership mềm
        all_distances = cdist(X_test_norm, cntr_norm)
        U_test = fuzzy_membership_from_distances(all_distances, m=FUZZINESS_M, eps=eps)
        test_cluster_assignments = np.argmin(all_distances, axis=1)  # vẫn giữ cho per-cluster metrics

        # 1) Expert GLOBAL (luôn có)
        global_model = clone(model_template)
        global_model.fit(X_train_scaled, y_train)
        if hasattr(global_model, "predict_proba"):
            proba_global = global_model.predict_proba(X_test_scaled)
        else:
            proba_global = np.eye(n_classes)[global_model.predict(X_test_scaled)]

        # Prior toàn cục
        counts_global = np.bincount(y_train, minlength=n_classes).astype(float)
        pi_global = counts_global / max(counts_global.sum(), 1.0)

        # 2) Train local experts (một lần) + prior adjust cho từng cụm
        local_models = {}           # i -> (model, reverse_map, local_global_classes)
        prior_adjustments = {}      # i -> vector (n_classes,) = pi_i_star / pi_global

        for i in range(n_clusters):
            # Local model nếu đủ điều kiện
            if trained_cluster_flags.get(i, False):
                cluster_indices_train = np.where(cluster_membership_train == i)[0]
                X_cluster_train = X_train_scaled[cluster_indices_train]
                y_cluster_train_global = y_train[cluster_indices_train]

                local_label_map = label_maps[i]
                reverse_map = {l: g for g, l in local_label_map.items()}
                y_cluster_train_local = np.array([local_label_map[g] for g in y_cluster_train_global])

                tmp_model = clone(model_template)
                try:
                    tmp_model.fit(X_cluster_train, y_cluster_train_local)
                    local_models[i] = (tmp_model, reverse_map, [reverse_map[l] for l in tmp_model.classes_])
                except Exception as e:
                    print(f"Warning: local fit failed for cluster {i}: {e}")
                    trained_cluster_flags[i] = False

            # Prior cụm + shrinkage về global
            cluster_indices_train = np.where(cluster_membership_train == i)[0]
            if cluster_indices_train.size > 0:
                y_ci = y_train[cluster_indices_train]
                counts_ci = np.bincount(y_ci, minlength=n_classes).astype(float)
                n_ci = counts_ci.sum()
                pi_ci = counts_ci / max(n_ci, 1.0)
            else:
                counts_ci = np.zeros(n_classes, dtype=float)
                n_ci = 0.0
                pi_ci = np.zeros(n_classes, dtype=float)

            pi_star = (n_ci * pi_ci + PRIOR_SMOOTH_TAU * pi_global) / max(n_ci + PRIOR_SMOOTH_TAU, 1.0)
            ratio = (pi_star + EPS_SAFE) / (pi_global + EPS_SAFE)  # vector (n_classes,)
            prior_adjustments[i] = ratio

        # END TOTAL TRAINING TIME MEASUREMENT (after clustering + global + local models training)
        time_total_training = time.perf_counter() - start_time_total_training
        print(f"Total training time (clustering + models): {time_total_training:.4f}s")

        # Update cluster_summary with total training time
        cluster_summary['Time_Total_Training_s'] = float(time_total_training)

        # 3) Hòa trộn MoE
        n_test = X_test_norm.shape[0]
        # Hiệu chỉnh prior theo cụm chính cho expert global
        main_cluster = np.argmax(U_test, axis=1)
        ratio_matrix = np.vstack([prior_adjustments[c] for c in main_cluster])  # (n_test, n_classes)
        proba_global_adj = proba_global * ratio_matrix
        proba_global_adj = proba_global_adj / (proba_global_adj.sum(axis=1, keepdims=True) + EPS_SAFE)

        # Khởi tạo mixture bằng GLOBAL đã hiệu chỉnh
        weights_global = np.full(n_test, GLOBAL_BASE_WEIGHT, dtype=float)
        mixture_sum = weights_global[:, None] * proba_global_adj
        weights_sum = weights_global.copy()

        # Cộng các expert LOCAL (nếu có), weight = membership ** LAMBDA_GATE
        for i, pack in local_models.items():
            tmp_model, reverse_map, local_global_classes = pack
            w_i = U_test[:, i] ** LAMBDA_GATE
            idx = np.where(w_i > MEMBERSHIP_THRESH)[0]
            if idx.size == 0:
                continue
            if hasattr(tmp_model, "predict_proba"):
                proba_local_sub = tmp_model.predict_proba(X_test_scaled[idx])
            else:
                proba_local_sub = np.eye(len(tmp_model.classes_))[tmp_model.predict(X_test_scaled[idx])]
            proba_local_sub_global = proba_local_to_global(proba_local_sub, local_global_classes, n_classes)

            mixture_sum[idx] += (w_i[idx, None] * proba_local_sub_global)
            weights_sum[idx] += w_i[idx]

        # Chuẩn hóa mixture
        y_score = mixture_sum / (weights_sum[:, None] + EPS_SAFE)
        y_pred = np.argmax(y_score, axis=1)

        time_total_prediction = time.perf_counter() - start_time_total_prediction
        time_per_sample_prediction = time_total_prediction / (X_test_norm.shape[0] if X_test_norm.shape[0] > 0 else 1.0)

        # Đo thời gian test inference hoàn chỉnh (tương tự moe_tradition_v3.py)
        # Bao gồm: chuẩn hóa data + tính distances + memberships + mixture + argmax
        test_inference_start = time.perf_counter()
        # Simulate complete test inference pipeline
        X_test_inf = normalize(X_test_scaled + eps, norm='l2', axis=1)  # normalize
        distances_inf = cdist(X_test_inf, cntr_norm)  # distances to centers
        U_test_inf = fuzzy_membership_from_distances(distances_inf, m=FUZZINESS_M, eps=eps)  # memberships
        # Calculate mixture (simplified version of the full pipeline)
        mixture_sum_inf = weights_global[:, None] * proba_global_adj
        weights_sum_inf = weights_global.copy()
        for i, pack in local_models.items():
            tmp_model, reverse_map, local_global_classes = pack
            w_i = U_test_inf[:, i] ** LAMBDA_GATE
            idx = np.where(w_i > MEMBERSHIP_THRESH)[0]
            if idx.size == 0:
                continue
            if hasattr(tmp_model, "predict_proba"):
                proba_local_sub = tmp_model.predict_proba(X_test_scaled[idx])
            else:
                proba_local_sub = np.eye(len(tmp_model.classes_))[tmp_model.predict(X_test_scaled[idx])]
            proba_local_sub_global = proba_local_to_global(proba_local_sub, local_global_classes, n_classes)
            mixture_sum_inf[idx] += (w_i[idx, None] * proba_local_sub_global)
            weights_sum_inf[idx] += w_i[idx]
        y_score_inf = mixture_sum_inf / (weights_sum_inf[:, None] + EPS_SAFE)
        y_pred_inf = np.argmax(y_score_inf, axis=1)  # final prediction
        test_time_sec = time.perf_counter() - test_inference_start
        test_time_ms_per_sample = 1000.0 * test_time_sec / max(1, len(y_test))

        print(f"Time assign 1 sample: {time_assign_single_point:.8f}s, time per sample full predict: {time_per_sample_prediction:.8f}s")
        print(f"Test inference time: {test_time_sec:.8f}s ({test_time_ms_per_sample:.3f}ms per sample)")

        # ---------------------
        # Metrics
        # ---------------------
        accuracy = metrics.accuracy_score(y_test, y_pred)
        report_labels_for_metrics = np.union1d(y_test, y_pred)

        precision_macro = metrics.precision_score(y_test, y_pred, labels=report_labels_for_metrics, average='macro', zero_division=0)
        recall_macro = metrics.recall_score(y_test, y_pred, labels=report_labels_for_metrics, average='macro', zero_division=0)
        f1_macro = metrics.f1_score(y_test, y_pred, labels=report_labels_for_metrics, average='macro', zero_division=0)

        precision_weighted = metrics.precision_score(y_test, y_pred, labels=report_labels_for_metrics, average='weighted', zero_division=0)
        recall_weighted = metrics.recall_score(y_test, y_pred, labels=report_labels_for_metrics, average='weighted', zero_division=0)
        f1_weighted = metrics.f1_score(y_test, y_pred, labels=report_labels_for_metrics, average='weighted', zero_division=0)

        print(f"Accuracy: {accuracy:.6f}, F1_macro: {f1_macro:.6f}, F1_weighted: {f1_weighted:.6f}")

        # Confusion + per-class TPR/FPR/DR
        labels_eval = np.union1d(y_test, y_pred)
        cm_eval = metrics.confusion_matrix(y_test, y_pred, labels=labels_eval)

        TP = np.diag(cm_eval).astype(float)
        FP = cm_eval.sum(axis=0) - TP
        FN = cm_eval.sum(axis=1) - TP
        TN = cm_eval.sum() - (TP + FP + FN)

        tpr_per_class = np.divide(TP, (TP + FN), out=np.zeros_like(TP), where=(TP + FN) != 0)
        fpr_per_class = np.divide(FP, (FP + TN), out=np.zeros_like(FP), where=(FP + TN) != 0)
        support_per_class = cm_eval.sum(axis=1).astype(float)

        TPR_Macro = np.nanmean(tpr_per_class)
        FPR_Macro = np.nanmean(fpr_per_class)
        TPR_Weighted = np.average(tpr_per_class, weights=np.where(support_per_class>0, support_per_class, 0))
        FPR_Weighted = np.average(fpr_per_class, weights=np.where(support_per_class>0, support_per_class, 0))

        names_eval = le.inverse_transform(labels_eval)
        df_fpr_tpr = pd.DataFrame({
            'class': names_eval,
            'support': support_per_class.astype(int),
            'TPR(Recall)': tpr_per_class,
            'FPR': fpr_per_class
        })
        df_fpr_tpr_path = os.path.join(current_result_dir, 'fpr_tpr_per_class.csv')
        df_fpr_tpr.to_csv(df_fpr_tpr_path, index=False, float_format='%.6f')

        benign_idx_eval = [i for i, name in enumerate(names_eval) if 'benign' in name.lower()]
        attack_idx_eval = [i for i in range(len(names_eval)) if i not in benign_idx_eval]
        attack_tp = TP[attack_idx_eval].sum() if len(attack_idx_eval)>0 else 0.0
        attack_total = support_per_class[attack_idx_eval].sum() if len(attack_idx_eval)>0 else 0.0
        DR_Attack = (attack_tp / attack_total) if attack_total > 0 else np.nan

        # ROC/AUC (one-vs-rest) cho các lớp hiện diện trong test
        classes_present = np.unique(y_test)
        try:
            y_test_bin = label_binarize(y_test, classes=classes_present)
        except Exception:
            y_test_bin = None

        y_score_present = y_score[:, classes_present] if y_test_bin is not None else None
        fpr_dict, tpr_dict, auc_dict = {}, {}, {}
        if y_test_bin is not None:
            for idx, c in enumerate(classes_present):
                try:
                    fpr_i, tpr_i, _ = metrics.roc_curve(y_test_bin[:, idx], y_score_present[:, idx])
                    auc_i = metrics.auc(fpr_i, tpr_i)
                except Exception:
                    fpr_i, tpr_i, auc_i = np.array([0,1]), np.array([0,1]), np.nan
                fpr_dict[c], tpr_dict[c], auc_dict[c] = fpr_i, tpr_i, auc_i

            fpr_micro, tpr_micro, _ = metrics.roc_curve(y_test_bin.ravel(), y_score_present.ravel())
            auc_micro = metrics.auc(fpr_micro, tpr_micro)

            all_fpr = np.unique(np.concatenate([fpr_dict[c] for c in classes_present]))
            mean_tpr = np.zeros_like(all_fpr)
            for c in classes_present:
                # np.interp có thể không có left/right ở numpy cũ; nếu lỗi, bỏ 2 tham số đó
                try:
                    mean_tpr += np.interp(all_fpr, fpr_dict[c], tpr_dict[c], left=0, right=1)
                except TypeError:
                    mean_tpr += np.interp(all_fpr, fpr_dict[c], tpr_dict[c])
            mean_tpr /= len(classes_present)
            auc_macro = metrics.auc(all_fpr, mean_tpr)
        else:
            auc_micro = np.nan
            auc_macro = np.nan
            fpr_micro = tpr_micro = all_fpr = mean_tpr = np.array([0,1])

        df_auc = pd.DataFrame({
            'class': le.inverse_transform(classes_present),
            'AUC': [auc_dict.get(c, np.nan) for c in classes_present]
        })
        auc_csv_path = os.path.join(current_result_dir, 'roc_auc_per_class.csv')
        df_auc.to_csv(auc_csv_path, index=False, float_format='%.6f')

        # plot ROC macro/micro
        try:
            plt.figure()
            plt.plot(fpr_micro, tpr_micro, linestyle=':', label=f'micro-average ROC (AUC={auc_micro:.3f})')
            plt.plot(all_fpr, mean_tpr, label=f'macro-average ROC (AUC={auc_macro:.3f})')
            plt.plot([0,1],[0,1], linestyle='--', label='Chance')
            plt.xlim([0.0,1.0]); plt.ylim([0.0,1.05])
            plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
            plt.title(f'ROC – {model_name} – {n_clusters} clusters (COSINE + MoE)')
            plt.legend(loc='lower right'); plt.tight_layout()
            roc_img_path = os.path.join(current_result_dir, 'roc_macro_micro.png')
            plt.savefig(roc_img_path, dpi=200); plt.close()
        except Exception as e:
            print("Warning: ROC plot failed:", e)

        # Per-cluster metrics on test
        per_cluster_metrics = []
        for i in range(n_clusters):
            idxs = np.where(test_cluster_assignments == i)[0]
            if idxs.size == 0:
                per_cluster_metrics.append({
                    'cluster': int(i),
                    'n_test_samples': 0,
                    'accuracy': np.nan,
                    'precision_macro': np.nan,
                    'recall_macro': np.nan,
                    'f1_macro': np.nan,
                    'precision_weighted': np.nan,
                    'recall_weighted': np.nan,
                    'f1_weighted': np.nan,
                    'trained_model_available': bool(trained_cluster_flags.get(i, False))
                })
                continue
            y_true_sub = y_test[idxs]
            y_pred_sub = y_pred[idxs]
            acc_i = metrics.accuracy_score(y_true_sub, y_pred_sub)
            pr_macro_i = metrics.precision_score(y_true_sub, y_pred_sub, average='macro', zero_division=0)
            rc_macro_i = metrics.recall_score(y_true_sub, y_pred_sub, average='macro', zero_division=0)
            f1_macro_i = metrics.f1_score(y_true_sub, y_pred_sub, average='macro', zero_division=0)
            pr_w_i = metrics.precision_score(y_true_sub, y_pred_sub, average='weighted', zero_division=0)
            rc_w_i = metrics.recall_score(y_true_sub, y_pred_sub, average='weighted', zero_division=0)
            f1_w_i = metrics.f1_score(y_true_sub, y_pred_sub, average='weighted', zero_division=0)

            per_cluster_metrics.append({
                'cluster': int(i),
                'n_test_samples': int(idxs.size),
                'accuracy': float(acc_i),
                'precision_macro': float(pr_macro_i),
                'recall_macro': float(rc_macro_i),
                'f1_macro': float(f1_macro_i),
                'precision_weighted': float(pr_w_i),
                'recall_weighted': float(rc_w_i),
                'f1_weighted': float(f1_w_i),
                'trained_model_available': bool(trained_cluster_flags.get(i, False))
            })

        df_cluster_test = pd.DataFrame(per_cluster_metrics)
        df_cluster_both = df_cluster_train.merge(df_cluster_test, on='cluster', how='outer')
        cluster_summary_csv = os.path.join(current_result_dir, 'cluster_training_summary.csv')
        df_cluster_both.to_csv(cluster_summary_csv, index=False, float_format='%.6f')

        # Save textual report
        report_path = os.path.join(current_result_dir, 'classification_results.txt')
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(f"KẾT QUẢ VỚI {n_clusters} CỤM (FCM - COSINE + MoE) SỬ DỤNG MÔ HÌNH {model_name}\n")
            f.write("="*80 + "\n\n")
            f.write("I. PHÂN CỤM\n")
            f.write(f" - PC: {pc_metric:.6f}, CE: {ce_metric:.6f}, XB: {xb_metric:.6f}\n")
            f.write(f" - Thời gian tìm tâm cụm: {time_clustering_train:.6f} s\n")
            f.write(f" - Thời gian training tổng: {time_total_training:.6f} s\n\n")
            f.write("II. PHÂN LOẠI (TEST)\n")
            f.write(f" - Accuracy: {accuracy:.6f}\n")
            f.write(f" - Precision (Macro): {precision_macro:.6f}\n")
            f.write(f" - Recall (Macro): {recall_macro:.6f}\n")
            f.write(f" - F1 (Macro): {f1_macro:.6f}\n")
            f.write(f" - Precision (Weighted): {precision_weighted:.6f}\n")
            f.write(f" - Recall (Weighted): {recall_weighted:.6f}\n")
            f.write(f" - F1 (Weighted): {f1_weighted:.6f}\n")
            f.write(f" - Time total training: {time_total_training:.8f} s\n")
            f.write(f" - Time assign one sample: {time_assign_single_point:.8f} s\n")
            f.write(f" - Time total per sample: {time_per_sample_prediction:.8f} s\n")
            f.write(f" - Test inference time: {test_time_sec:.8f} s ({test_time_ms_per_sample:.3f} ms per sample)\n\n")
            f.write("III. FPR/TPR/DR\n")
            f.write(f" - TPR Macro: {TPR_Macro:.6f}, FPR Macro: {FPR_Macro:.6f}\n")
            f.write(f" - TPR Weighted: {TPR_Weighted:.6f}, FPR Weighted: {FPR_Weighted:.6f}\n")
            f.write(f" - DR Attack: {DR_Attack}\n")
            f.write(f" -> per-class TPR/FPR: {df_fpr_tpr_path}\n\n")
            f.write("IV. ROC/AUC\n")
            f.write(f" - AUC Micro: {auc_micro if not np.isnan(auc_micro) else 'nan'}\n")
            f.write(f" - AUC Macro: {auc_macro if not np.isnan(auc_macro) else 'nan'}\n")
            f.write(f" -> per-class AUC: {auc_csv_path}\n\n")
            f.write("V. PER-CLUSTER SUMMARY\n")
            f.write(f" -> {cluster_summary_csv}\n\n")
            f.write("="*40 + "\n\n")
        print(f"Saved text report: {report_path}")

        # Save some summary fields
        cluster_summary[f'Accuracy_{model_name}'] = float(accuracy)
        cluster_summary[f'Time_Assign_One_Sample_s_{model_name}'] = float(time_assign_single_point)
        cluster_summary[f'Time_Total_Predict_Per_Sample_s_{model_name}'] = float(time_per_sample_prediction)
        cluster_summary[f'Precision_Macro_{model_name}'] = float(precision_macro)
        cluster_summary[f'Recall_Macro_{model_name}'] = float(recall_macro)
        cluster_summary[f'F1_Macro_{model_name}'] = float(f1_macro)
        cluster_summary[f'Precision_Weighted_{model_name}'] = float(precision_weighted)
        cluster_summary[f'Recall_Weighted_{model_name}'] = float(recall_weighted)
        cluster_summary[f'F1_Weighted_{model_name}'] = float(f1_weighted)
        cluster_summary[f'TPR_Macro_{model_name}'] = float(TPR_Macro)
        cluster_summary[f'FPR_Macro_{model_name}'] = float(FPR_Macro)
        cluster_summary[f'TPR_Weighted_{model_name}'] = float(TPR_Weighted)
        cluster_summary[f'FPR_Weighted_{model_name}'] = float(FPR_Weighted)
        cluster_summary[f'DR_Attack_{model_name}'] = float(DR_Attack) if not np.isnan(DR_Attack) else np.nan
        cluster_summary[f'AUC_Macro_{model_name}'] = float(auc_macro) if not np.isnan(auc_macro) else np.nan
        cluster_summary[f'AUC_Micro_{model_name}'] = float(auc_micro) if not np.isnan(auc_micro) else np.nan
        cluster_summary[f'Test_Time_Sec_{model_name}'] = float(test_time_sec)
        cluster_summary[f'Test_Time_MS_Per_Sample_{model_name}'] = float(test_time_ms_per_sample)

        # Per-cluster plots và confusion
        try:
            plot_bar_per_cluster(df_cluster_both, current_result_dir, model_name, n_clusters)
        except Exception as e:
            print("Warning: per-cluster bar plot failed:", e)
        try:
            plot_confusion_matrix(cm_eval, names_eval, current_result_dir, model_name, n_clusters, normalize=False)
            plot_confusion_matrix(cm_eval, names_eval, current_result_dir, model_name, n_clusters, normalize=True)
        except Exception as e:
            print("Warning: confusion matrix plot failed:", e)

        # Save best cluster info (by accuracy và by F1-macro)
        try:
            df_cb_tested = df_cluster_both[df_cluster_both['n_test_samples'] > 0].copy()
            best_acc_row = None
            best_f1_row = None
            if not df_cb_tested.empty:
                best_acc_row = df_cb_tested.loc[df_cb_tested['accuracy'].idxmax()].to_dict()
                best_f1_row = df_cb_tested.loc[df_cb_tested['f1_macro'].idxmax()].to_dict()
            best_txt_path = os.path.join(current_result_dir, f'best_clusters_{model_name}.txt')
            with open(best_txt_path, 'w', encoding='utf-8') as bf:
                bf.write(f"Best clusters for model {model_name} with K={n_clusters} (COSINE + MoE)\n\n")
                if best_acc_row is not None:
                    bf.write("By Accuracy (per-cluster on test samples):\n")
                    bf.write(str(best_acc_row) + "\n\n")
                else:
                    bf.write("No test samples assigned to any cluster.\n\n")
                if best_f1_row is not None:
                    bf.write("By F1-macro (per-cluster on test samples):\n")
                    bf.write(str(best_f1_row) + "\n\n")
            print(f"Saved best cluster info: {best_txt_path}")
        except Exception as e:
            print("Warning: saving best-cluster info failed:", e)

    overall_results.append(cluster_summary)

# ---------------------------
# End K sweep: overall reports & plots
# ---------------------------
if overall_results:
    df_report = pd.DataFrame(overall_results)

    # Weighted summary (existing fields)
    weighted_acc_cols = sorted([col for col in df_report.columns if 'Accuracy' in col and 'Time' not in col])
    weighted_prf_cols = sorted([col for col in df_report.columns if 'Weighted' in col])
    time_cols = sorted([col for col in df_report.columns if 'Time' in col])
    auc_cols = sorted([col for col in df_report.columns if 'AUC_' in col])

    weighted_report_cols = ['Số cụm', 'PC', 'CE', 'XB'] + weighted_acc_cols + weighted_prf_cols + auc_cols + time_cols
    weighted_report_cols_exist = [col for col in weighted_report_cols if col in df_report.columns]
    df_report_weighted = df_report[weighted_report_cols_exist]
    overall_report_weighted_path = os.path.join(RESULT_DIR, 'overall_summary_report_weighted.csv')
    df_report_weighted.to_csv(overall_report_weighted_path, index=False, float_format='%.6f')
    print(f"Saved overall weighted report: {overall_report_weighted_path}")

    # Full report
    acc_cols = sorted([col for col in df_report.columns if 'Accuracy' in col and 'Time' not in col])
    macro_cols = sorted([col for col in df_report.columns if 'Macro' in col])
    full_report_cols = ['Số cụm', 'PC', 'CE', 'XB'] + acc_cols + macro_cols + weighted_prf_cols + auc_cols + time_cols
    full_report_cols_exist = [col for col in full_report_cols if col in df_report.columns]
    df_report_full = df_report[full_report_cols_exist]
    overall_report_full_path = os.path.join(RESULT_DIR, 'overall_summary_report_full.csv')
    df_report_full.to_csv(overall_report_full_path, index=False, float_format='%.6f')
    print(f"Saved overall full report: {overall_report_full_path}")

    # Summary plots vs K
    try:
        plot_over_k(df_report_full, RESULT_DIR)
        print("Saved summary plots vs K")
    except Exception as e:
        print("Warning: summary plots vs K failed:", e)
else:
    print("No overall results to report.")

print("\nALL DONE — experiment finished.")
def plot_over_k(df, outdir):
    plt.figure()
    for c in sorted([col for col in df.columns if 'Accuracy' in col and 'Time' not in col]):
        plt.plot(df['Số cụm'], df[c], marker='o', label=c.replace('Accuracy_','Acc:'))
    plt.xlabel('K (số cụm)'); plt.ylabel('Accuracy'); plt.title('Accuracy (per model) vs K')
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(outdir, 'accuracy_per_model_vs_K.png'), dpi=200); plt.close()

    plt.figure()
    for c in sorted([col for col in df.columns if 'F1_Weighted' in col]):
        plt.plot(df['Số cụm'], df[c], marker='o', label=c.replace('F1_Weighted_','F1w:'))
    plt.xlabel('K (số cụm)'); plt.ylabel('F1 (Weighted)'); plt.title('F1-weighted (per model) vs K')
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(outdir, 'f1_weighted_per_model_vs_K.png'), dpi=200); plt.close()

    f1_macro_cols = sorted([col for col in df.columns if 'F1_Macro' in col])
    if f1_macro_cols:
        plt.figure()
        for c in f1_macro_cols:
            plt.plot(df['Số cụm'], df[c], marker='o', label=c.replace('F1_Macro_','F1m:'))
        plt.xlabel('K (số cụm)'); plt.ylabel('F1 (Macro)'); plt.title('F1-macro (per model) vs K')
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(outdir, 'f1_macro_per_model_vs_K.png'), dpi=200); plt.close()  